In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import sympy as sym

from rotation.symbolic.general import COUPLING, ROTATED, BARE, create_3d_rotation
from rotation.symbolic.pseudo import create_mix_equation, rotated_pseudo_meson_couplings, bare_pseudo_masses, rotated_pseudo_photon_coupling, bare_pseudo_kaon_couplings, rotated_pseudo_kaon_couplings

## Mixing Equation

In [ ]:
# Define some labels to increase clarity of output
rotate_2d_symbol = sym.MatrixSymbol(r"R_{2D}", 2, 2)
rotation_symbols = []
for name in [r"R_\alpha", r"R_\beta", r"R_\gamma"]:
    rotation_symbols.append(sym.MatrixSymbol(name, 3, 3))
rotate_symbol = sym.MatrixSymbol("R", 3, 3)
matrix_symbols = []
for name in [r"R^\text{T}", r"M_b", "M"]:
    matrix_symbols.append(sym.MatrixSymbol(name, 3, 3))

In [ ]:
rotate, rotations, rotate_2d = create_3d_rotation()
pseudo_equation, pseudo_matrices = create_mix_equation(rotate)

In [ ]:
# Rotation
print("The 2D rotation matrix is:")
sym.Eq(rotate_2d_symbol, rotate_2d, evaluate=False)
print("Which is used to create 3 different 3D rotations:")
for symbol, rot in zip(rotation_symbols, rotations):
    sym.Eq(symbol, rot, evaluate=False)
print("And their multiplication creates the general 3D rotation matrix:")
sym.Eq(
    rotate_symbol,
    sym.MatMul(*rotation_symbols, evaluate=False),
    evaluate=False,
)
sym.Eq(
    rotate_symbol, sym.MatMul(*rotations, evaluate=False), evaluate=False
)

In [ ]:
print("So our four matrices are:")
for s, mat in zip([rotate_symbol]+matrix_symbols, [rotate]+list(pseudo_matrices)):
    sym.Eq(s, mat, evaluate=False)
print("Which are all used in the mixing equation:")
sym.Eq(
    matrix_symbols[2],
    sym.MatMul(rotate_symbol, *matrix_symbols[1::-1]),
    evaluate=False,
)
pseudo_equation
sym.Eq(
    pseudo_equation.lhs,
    1*pseudo_equation.rhs,
    evaluate=False,
)

In [ ]:
print("The following are the off-diagonal bare mass terms:")
for s, g in zip([BARE[k] for k in ["pi0-phi", "eta-phi"]], bare_pseudo_masses()):
    sym.Eq(s, g, evaluate=False)

## Couplings

In [ ]:
print("The following are the rotated dark matter to meson couplings:")
for s, g in zip([ROTATED[k] for k in ["pi-chi", "phi-chi", "eta-chi"]], rotated_pseudo_meson_couplings()):
    sym.Eq(s, g, evaluate=False)

In [ ]:
print("The following is the rotated pseudo scalar mediator to photon coupling (having factored out the un-rotated bare pion to photon and eta to photon couplings):")
sym.Eq(ROTATED["phi-gamma"], rotated_pseudo_photon_coupling(), evaluate=False)

In [ ]:
print("The following are the un-rotated bare pion-pion-kaon and eta-pion-kaon couplings (mediator-pion-kaon is zero):")
for s, g in zip([COUPLING[k] for k in ["pi-pi-K", "eta-pi-K"]], bare_pseudo_kaon_couplings()):
    sym.Eq(s, g, evaluate=False)

In [ ]:
print("The following are the rotated kaon couplings:")
for s, g in zip([ROTATED[k] for k in ["pi-pi-K", "phi-pi-K", "eta-pi-K"]], rotated_pseudo_kaon_couplings()):
    sym.Eq(s, g, evaluate=False)

In [ ]:
bares = list(bare_pseudo_kaon_couplings())
bares.insert(1,0)
print("Substituting the un-rotated couplings into the rotated kaon couplings:")
for s, g in zip([ROTATED[k] for k in ["pi-pi-K", "phi-pi-K", "eta-pi-K"]], rotated_pseudo_kaon_couplings()):
    subbed = g.subs({COUPLING[k]:v for k,v in zip(["pi-pi-K", "phi-pi-K", "eta-pi-K"], bares)})
    sym.Eq(s, subbed, evaluate=False)